# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 2: The Classic Heuristic (Constructive Base-Stock Optimization)

### Key assumptions
- Demand follows a normal distribution with known parameters
- Lead time is constant and known
- Review periods can be any positive value (not limited to integers)
- Base-stock level is optimized for each review period
- Cost parameters remain constant over time

### Approach (step-by-step)
1. **Grid Search**: Systematically evaluate different review periods
2. **Base-Stock Calculation**: For each review period, compute optimal base-stock level
3. **Cost Evaluation**: Calculate total expected cost for each (R,S) combination
4. **Optimization**: Select the review period with minimum total cost
5. **Simulation**: Verify results through Monte Carlo simulation

### What to look for in the results
- Optimal review period that may differ from analytical assumptions
- Cost savings compared to fixed review period approach
- Service level achievement through simulation
- Robustness of the heuristic solution

### Concrete example (from the source)
MedSupply Distribution syringe inventory:
- Weekly demand: μ = 12,000 units, σ = 2,400 units
- Lead time: L = 0.5 weeks
- Service level: α = 0.95
- Costs: h = $0.15/unit/week, K = $75/order, p = $8/unit

Expected output:
- Optimal Review Period: ~1.30 weeks
- Optimal Base-Stock Level: ~23,156 units
- Minimum Total Cost: ~$2,284.73 per week

### Why this Tier exists vs Tier 1
The heuristic approach provides practical advantages over pure analytical methods:
- **Flexibility**: Can handle non-standard review periods (not just integers)
- **Robustness**: Works with discrete cost evaluation rather than continuous approximations
- **Practicality**: More suitable for real-world implementation where review schedules may be constrained
- **Verification**: Includes simulation to validate analytical results

### Pros / Cons vs Tier 1
**Pros:**
- More flexible in handling various review period constraints
- Includes simulation verification for robustness
- Can handle discrete decision variables better
- More intuitive for practitioners to understand and implement

**Cons:**
- Computationally more intensive than analytical solution
- May not guarantee global optimality if grid is not fine enough
- Still relies on same assumptions as Tier 1
- Limited to single-objective optimization

### When to use this Tier
- When review periods are constrained to specific values
- When you need simulation verification of analytical results
- For practical implementation in real-world systems
- When discrete decision variables are important
- When robustness is more important than theoretical optimality

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from math import sqrt, inf
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


Libraries imported successfully!


Libraries imported successfully!


Libraries imported successfully!


In [2]:
class BaseStockHeuristicOptimizer:
    """
    Constructive Heuristic for Periodic Review Base-Stock Policy Optimization
    
    This class implements a grid search heuristic that systematically evaluates
    different review periods and calculates the corresponding optimal base-stock
    levels, selecting the combination that minimizes total expected costs.
    """
    
    def __init__(self, demand_mean, demand_std, lead_time, holding_cost, 
                 ordering_cost, stockout_cost, service_level):
        """
        Initialize the heuristic optimizer with problem parameters.
        """
        self.mu = demand_mean
        self.sigma = demand_std
        self.L = lead_time
        self.h = holding_cost
        self.K = ordering_cost
        self.p = stockout_cost
        self.alpha = service_level
        self.z_alpha = norm.ppf(service_level)
        
        print(f"Initialized Heuristic Optimizer:")
        print(f"  Demand: N({self.mu:,}, {self.sigma:,}²)")
        print(f"  Lead time: {self.L} weeks")
        print(f"  Service level: {self.alpha:.3f}")
        print(f"  Costs: h=${self.h}, K=${self.K}, p=${self.p}")
    
    def calculate_base_stock_level(self, review_period):
        """
        Calculate optimal base-stock level for given review period.
        Uses the same formula as Tier 1 but applied within heuristic framework.
        """
        protection_interval = review_period + self.L
        expected_demand = self.mu * protection_interval
        demand_std = self.sigma * sqrt(protection_interval)
        base_stock = expected_demand + self.z_alpha * demand_std
        
        return base_stock, protection_interval, expected_demand, demand_std
    
    def calculate_total_cost(self, review_period, base_stock_level):
        """
        Calculate expected total cost for given parameters.
        Uses discrete cost evaluation suitable for heuristic optimization.
        """
        protection_interval = review_period + self.L
        
        # Ordering cost per period
        ordering_cost = self.K / review_period
        
        # Expected inventory components
        cycle_stock = (self.mu * review_period) / 2
        safety_stock = self.z_alpha * self.sigma * sqrt(protection_interval)
        expected_inventory = cycle_stock + safety_stock
        
        # Holding cost
        holding_cost = self.h * expected_inventory
        
        # Stockout cost using loss function
        demand_std = self.sigma * sqrt(protection_interval)
        normalized_shortage = (base_stock_level - self.mu * protection_interval) / demand_std
        loss_function = demand_std * (norm.pdf(normalized_shortage) - 
                                     normalized_shortage * (1 - norm.cdf(normalized_shortage)))
        stockout_cost = (self.p / review_period) * loss_function
        
        total_cost = ordering_cost + holding_cost + stockout_cost
        
        return total_cost, ordering_cost, holding_cost, stockout_cost
    
    def optimize_policy(self, min_review=0.1, max_review=4.0, step=0.01):
        """
        Grid search optimization over review periods.
        
        Parameters:
        - min_review: Minimum review period to consider
        - max_review: Maximum review period to consider
        - step: Step size for grid search (smaller = more precise)
        """
        best_cost = float('inf')
        best_review = None
        best_base_stock = None
        results = []
        
        print(f"\nStarting grid search: R ∈ [{min_review}, {max_review}] with step {step}")
        
        for R in np.arange(min_review, max_review + step, step):
            # Calculate optimal base-stock for this review period
            S = self.calculate_base_stock_level(R)[0]
            
            # Calculate total cost
            total_cost, ord_cost, hold_cost, stock_cost = self.calculate_total_cost(R, S)
            
            # Store results
            result = {
                'review_period': R,
                'base_stock_level': S,
                'total_cost': total_cost,
                'ordering_cost': ord_cost,
                'holding_cost': hold_cost,
                'stockout_cost': stock_cost
            }
            results.append(result)
            
            # Update best solution
            if total_cost < best_cost:
                best_cost = total_cost
                best_review = R
                best_base_stock = S
        
        return {
            'optimal_review_period': best_review,
            'optimal_base_stock_level': best_base_stock,
            'minimum_total_cost': best_cost,
            'all_results': results
        }
    
    def simulate_policy(self, review_period, base_stock_level, num_periods=52, seed=42):
        """
        Monte Carlo simulation to verify heuristic results.
        
        Parameters:
        - review_period: Review period in weeks
        - base_stock_level: Base-stock level
        - num_periods: Number of periods to simulate
        - seed: Random seed for reproducibility
        """
        np.random.seed(seed)
        
        inventory_levels = []
        order_quantities = []
        demands = []
        stockout_indicators = []
        
        # Initialize inventory at base-stock level
        current_inventory = base_stock_level
        
        # Pipeline orders (simplified - using discrete time steps)
        pipeline_length = int(self.L * 10)  # Convert to discrete steps
        pipeline_orders = [0] * pipeline_length
        
        for period in range(num_periods):
            # Generate demand
            demand = max(0, np.random.normal(self.mu, self.sigma))
            demands.append(demand)
            
            # Receive orders due this period
            if period >= pipeline_length:
                current_inventory += pipeline_orders[period % pipeline_length]
                pipeline_orders[period % pipeline_length] = 0
            
            # Satisfy demand
            shortage = max(0, demand - current_inventory)
            current_inventory = max(0, current_inventory - demand)
            inventory_levels.append(current_inventory)
            stockout_indicators.append(1 if shortage > 0 else 0)
            
            # Place order at review periods
            if period % int(review_period * 10) == 0:
                pipeline_inventory = sum(pipeline_orders)
                inventory_position = current_inventory + pipeline_inventory
                order_quantity = max(0, base_stock_level - inventory_position)
                order_quantities.append(order_quantity)
                
                # Schedule delivery
                delivery_period = (period + pipeline_length) % len(pipeline_orders)
                pipeline_orders[delivery_period] += order_quantity
            else:
                order_quantities.append(0)
        
        # Calculate performance metrics
        avg_inventory = np.mean(inventory_levels)
        stockout_periods = sum(stockout_indicators)
        service_level = 1 - (stockout_periods / num_periods)
        total_orders = sum(order_quantities)
        avg_order_size = np.mean([q for q in order_quantities if q > 0]) if total_orders > 0 else 0
        
        return {
            'inventory_levels': inventory_levels,
            'order_quantities': order_quantities,
            'demands': demands,
            'avg_inventory': avg_inventory,
            'stockout_periods': stockout_periods,
            'service_level': service_level,
            'total_orders': total_orders,
            'avg_order_size': avg_order_size
        }

print("BaseStockHeuristicOptimizer class defined successfully!")

BaseStockHeuristicOptimizer class defined successfully!


BaseStockHeuristicOptimizer class defined successfully!


BaseStockHeuristicOptimizer class defined successfully!


BaseStockHeuristicOptimizer class defined successfully!


BaseStockHeuristicOptimizer class defined successfully!


In [ ]:
# Initialize the heuristic optimizer with MedSupply data
heuristic_optimizer = BaseStockHeuristicOptimizer(
    demand_mean=12000,      # μ = 12,000 units/week
    demand_std=2400,        # σ = 2,400 units/week
    lead_time=0.5,          # L = 0.5 weeks
    holding_cost=0.15,      # h = $0.15/unit/week
    ordering_cost=75,       # K = $75/order
    stockout_cost=8,        # p = $8/unit
    service_level=0.95      # α = 95% service level
)

In [ ]:
# Run the heuristic optimization
heuristic_result = heuristic_optimizer.optimize_policy(
    min_review=0.5, 
    max_review=3.0, 
    step=0.01
)

print("\n" + "="*60)
print("HEURISTIC OPTIMIZATION RESULTS")
print("="*60)
print(f"Optimal Review Period: {heuristic_result['optimal_review_period']:.2f} weeks")
print(f"Optimal Base-Stock Level: {heuristic_result['optimal_base_stock_level']:,.0f} units")
print(f"Minimum Total Cost: ${heuristic_result['minimum_total_cost']:,.2f} per week")
print()

# Get cost breakdown for optimal solution
optimal_R = heuristic_result['optimal_review_period']
optimal_S = heuristic_result['optimal_base_stock_level']
total_cost, ord_cost, hold_cost, stock_cost = heuristic_optimizer.calculate_total_cost(optimal_R, optimal_S)

print("COST BREAKDOWN:")
print(f"Ordering Cost: ${ord_cost:,.2f}/week ({ord_cost/heuristic_result['minimum_total_cost']*100:.1f}%)")
print(f"Holding Cost: ${hold_cost:,.2f}/week ({hold_cost/heuristic_result['minimum_total_cost']*100:.1f}%)")
print(f"Stockout Cost: ${stock_cost:,.2f}/week ({stock_cost/heuristic_result['minimum_total_cost']*100:.1f}%)")
print()

# Compare with Tier 1 analytical solution (1-week review)
tier1_cost = 2328.06  # From Tier 1 results
cost_savings = tier1_cost - heuristic_result['minimum_total_cost']
cost_savings_pct = (cost_savings / tier1_cost) * 100

print("COMPARISON WITH TIER 1 (1-week review):")
print(f"Tier 1 Cost: ${tier1_cost:,.2f}/week")
print(f"Heuristic Cost: ${heuristic_result['minimum_total_cost']:,.2f}/week")
print(f"Cost Savings: ${cost_savings:,.2f}/week ({cost_savings_pct:.1f}% reduction)")
print(f"Review Period Change: 1.00 → {heuristic_result['optimal_review_period']:.2f} weeks")

In [ ]:
# Convert results to DataFrame for analysis
results_df = pd.DataFrame(heuristic_result['all_results'])

# Find top 5 solutions
top_solutions = results_df.nsmallest(5, 'total_cost').copy()
top_solutions['cost_pct_diff'] = ((top_solutions['total_cost'] - top_solutions['total_cost'].min()) / 
                                   top_solutions['total_cost'].min() * 100)

print("TOP 5 SOLUTIONS FROM HEURISTIC SEARCH:")
print("="*80)
display_cols = ['review_period', 'base_stock_level', 'total_cost', 'ordering_cost', 
                'holding_cost', 'stockout_cost']
print(top_solutions[display_cols].round(2).to_string(index=False))

print("\n" + "="*50)
print("COST SENSITIVITY AROUND OPTIMAL SOLUTION:")
print("="*50)

# Analyze cost sensitivity around optimal solution
optimal_R = heuristic_result['optimal_review_period']
sensitivity_range = 0.3
sensitivity_results = results_df[
    (results_df['review_period'] >= optimal_R - sensitivity_range) & 
    (results_df['review_period'] <= optimal_R + sensitivity_range)
].copy()

sensitivity_results['cost_increase_pct'] = (
    (sensitivity_results['total_cost'] - heuristic_result['minimum_total_cost']) / 
    heuristic_result['minimum_total_cost'] * 100
)

print(f"Review Period Range: [{optimal_R - sensitivity_range:.2f}, {optimal_R + sensitivity_range:.2f}] weeks")
print(f"Cost Range: ${sensitivity_results['total_cost'].min():.2f} - ${sensitivity_results['total_cost'].max():.2f}")
print(f"Maximum Cost Increase: {sensitivity_results['cost_increase_pct'].max():.1f}%")

# Find flat region (within 1% of optimal)
flat_region = sensitivity_results[sensitivity_results['cost_increase_pct'] <= 1.0]
if len(flat_region) > 0:
    print(f"Flat Region (≤1% cost increase): {flat_region['review_period'].min():.2f} - {flat_region['review_period'].max():.2f} weeks")

In [ ]:
# Run Monte Carlo simulation to verify heuristic results
print("MONTE CARLO SIMULATION VERIFICATION")
print("="*50)

simulation_result = heuristic_optimizer.simulate_policy(
    review_period=heuristic_result['optimal_review_period'],
    base_stock_level=heuristic_result['optimal_base_stock_level'],
    num_periods=52,
    seed=42
)

print(f"Simulation Results (52 weeks):")
print(f"Average Inventory: {simulation_result['avg_inventory']:,.0f} units")
print(f"Service Level: {simulation_result['service_level']:.4f} ({simulation_result['service_level']*100:.2f}%)")
print(f"Stockout Periods: {simulation_result['stockout_periods']} out of 52 weeks")
print(f"Total Orders Placed: {simulation_result['total_orders']}")
print(f"Average Order Size: {simulation_result['avg_order_size']:,.0f} units")
print()

# Calculate simulated costs
sim_holding_cost = simulation_result['avg_inventory'] * heuristic_optimizer.h
sim_ordering_cost = simulation_result['total_orders'] * heuristic_optimizer.K / 52
sim_stockout_cost = simulation_result['stockout_periods'] * heuristic_optimizer.p * 2400 / 52  # Approximation
sim_total_cost = sim_holding_cost + sim_ordering_cost + sim_stockout_cost

print("SIMULATED COSTS:")
print(f"Holding Cost: ${sim_holding_cost:,.2f}/week")
print(f"Ordering Cost: ${sim_ordering_cost:,.2f}/week")
print(f"Stockout Cost: ${sim_stockout_cost:,.2f}/week")
print(f"Total Simulated Cost: ${sim_total_cost:,.2f}/week")
print()

# Compare analytical vs simulated
analytical_cost = heuristic_result['minimum_total_cost']
cost_diff = abs(sim_total_cost - analytical_cost)
cost_diff_pct = (cost_diff / analytical_cost) * 100

print("VERIFICATION RESULTS:")
print(f"Analytical Cost: ${analytical_cost:,.2f}/week")
print(f"Simulated Cost: ${sim_total_cost:,.2f}/week")
print(f"Difference: ${cost_diff:,.2f}/week ({cost_diff_pct:.1f}%)")
print(f"Service Level Target: 95.0%")
print(f"Achieved Service Level: {simulation_result['service_level']*100:.2f}%")

if cost_diff_pct < 5:
    print("✓ SIMULATION VERIFICATION PASSED (Cost difference < 5%)")
else:
    print("⚠ SIMULATION VERIFICATION WARNING (Cost difference > 5%)")

In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Heuristic Base-Stock Policy Optimization', fontsize=16, fontweight='bold')

# Plot 1: Total Cost vs Review Period
ax1.plot(results_df['review_period'], results_df['total_cost'], 
         'b-', linewidth=2, alpha=0.7, label='Total Cost')
ax1.axvline(x=heuristic_result['optimal_review_period'], color='red', 
           linestyle='--', alpha=0.8, label=f'Optimal R = {heuristic_result["optimal_review_period"]:.2f}')
ax1.axhline(y=heuristic_result['minimum_total_cost'], color='red', 
           linestyle='--', alpha=0.5, label=f'Min Cost = ${heuristic_result["minimum_total_cost"]:.0f}')
ax1.set_xlabel('Review Period (weeks)')
ax1.set_ylabel('Total Cost ($/week)')
ax1.set_title('Total Cost vs Review Period')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Cost Components at Optimal Solution
optimal_data = results_df[results_df['review_period'] == heuristic_result['optimal_review_period']].iloc[0]
costs = [optimal_data['ordering_cost'], optimal_data['holding_cost'], optimal_data['stockout_cost']]
labels = ['Ordering', 'Holding', 'Stockout']
colors = ['#ff9999', '#66b3ff', '#99ff99']

ax2.pie(costs, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title(f'Cost Breakdown\n(R = {heuristic_result["optimal_review_period"]:.2f} weeks)')

# Plot 3: Base-Stock Level vs Review Period
ax3.plot(results_df['review_period'], results_df['base_stock_level'], 
         'g-', linewidth=2, markersize=0)
ax3.axvline(x=heuristic_result['optimal_review_period'], color='red', 
           linestyle='--', alpha=0.8, label=f'Optimal R = {heuristic_result["optimal_review_period"]:.2f}')
ax3.axhline(y=heuristic_result['optimal_base_stock_level'], color='red', 
           linestyle='--', alpha=0.5, label=f'Optimal S = {heuristic_result["optimal_base_stock_level"]:.0f}')
ax3.set_xlabel('Review Period (weeks)')
ax3.set_ylabel('Base-Stock Level (units)')
ax3.set_title('Base-Stock Level vs Review Period')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Cost Comparison with Tier 1
comparison_data = {
    'Method': ['Tier 1 (Analytical)', 'Tier 2 (Heuristic)'],
    'Review Period': [1.0, heuristic_result['optimal_review_period']],
    'Total Cost': [tier1_cost, heuristic_result['minimum_total_cost']],
    'Base-Stock Level': [22838, heuristic_result['optimal_base_stock_level']]
}
comparison_df = pd.DataFrame(comparison_data)

x = np.arange(len(comparison_df['Method']))
width = 0.25

ax4.bar(x - width, comparison_df['Total Cost'], width, label='Total Cost', alpha=0.8)
ax4.bar(x, comparison_df['Review Period'] * 1000, width, label='Review Period (×1000)', alpha=0.8)
ax4.bar(x + width, comparison_df['Base-Stock Level'] / 50, width, label='Base-Stock (÷50)', alpha=0.8)

ax4.set_xlabel('Method')
ax4.set_ylabel('Value')
ax4.set_title('Tier 1 vs Tier 2 Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(comparison_df['Method'])
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete! Key insights:")
print(f"1. Optimal review period: {heuristic_result['optimal_review_period']:.2f} weeks")
print(f"2. Cost reduction vs Tier 1: {cost_savings_pct:.1f}%")
print(f"3. Heuristic finds better solution by optimizing review period")
print(f"4. Cost function shows clear convexity with unique optimum")

In [ ]:
# Detailed simulation analysis
print("DETAILED SIMULATION ANALYSIS")
print("="*50)

# Extract simulation time series
inventory_ts = simulation_result['inventory_levels']
demand_ts = simulation_result['demands']
order_ts = simulation_result['order_quantities']

# Calculate statistics
inventory_stats = {
    'mean': np.mean(inventory_ts),
    'std': np.std(inventory_ts),
    'min': np.min(inventory_ts),
    'max': np.max(inventory_ts),
    'median': np.median(inventory_ts)
}

demand_stats = {
    'mean': np.mean(demand_ts),
    'std': np.std(demand_ts),
    'min': np.min(demand_ts),
    'max': np.max(demand_ts),
    'median': np.median(demand_ts)
}

print("INVENTORY STATISTICS (52 weeks):")
for stat, value in inventory_stats.items():
    print(f"  {stat.capitalize()}: {value:,.0f} units")

print("\nDEMAND STATISTICS (52 weeks):")
for stat, value in demand_stats.items():
    print(f"  {stat.capitalize()}: {value:,.0f} units")

print(f"\nORDERING STATISTICS:")
print(f"  Total Orders: {simulation_result['total_orders']}")
print(f"  Average Order Size: {simulation_result['avg_order_size']:,.0f} units")
print(f"  Ordering Frequency: {simulation_result['total_orders']/52:.2f} orders/week")

# Calculate fill rate
total_demand = sum(demand_ts)
total_shortage = sum(max(0, d - inv) for d, inv in zip(demand_ts, inventory_ts))
fill_rate = (total_demand - total_shortage) / total_demand

print(f"\nSERVICE METRICS:")
print(f"  Service Level (no-stockout periods): {simulation_result['service_level']*100:.2f}%")
print(f"  Fill Rate (units delivered): {fill_rate*100:.2f}%")
print(f"  Stockout Frequency: {simulation_result['stockout_periods']}/52 weeks")

# Analyze inventory position at order times
order_times = [i for i, q in enumerate(order_ts) if q > 0]
if order_times:
    order_inventories = [inventory_ts[i] for i in order_times]
    print(f"\nINVENTORY AT ORDER TIMES:")
    print(f"  Average Inventory when Ordering: {np.mean(order_inventories):,.0f} units")
    print(f"  Order Times: {order_times[:5]}... (showing first 5)")

In [ ]:
# Summary and Conclusions
print("=" * 70)
print("PERIODIC REVIEW BASE-STOCK POLICY - TIER 2 SUMMARY")
print("=" * 70)
print()
print("HEURISTIC APPROACH OVERVIEW:")
print("• Grid search optimization over review periods")
print("• Base-stock level calculation for each review period")
print("• Monte Carlo simulation verification")
print("• Practical implementation considerations")
print()
print("OPTIMAL SOLUTION FOUND:")
print(f"• Review Period: {heuristic_result['optimal_review_period']:.2f} weeks")
print(f"• Base-Stock Level: {heuristic_result['optimal_base_stock_level']:,.0f} units")
print(f"• Total Cost: ${heuristic_result['minimum_total_cost']:,.2f}/week")
print()
print("PERFORMANCE IMPROVEMENT VS TIER 1:")
print(f"• Cost Reduction: {cost_savings_pct:.1f}% (${cost_savings:,.2f}/week)")
print(f"• Review Period Extension: 1.00 → {heuristic_result['optimal_review_period']:.2f} weeks")
print(f"• Base-Stock Adjustment: 22,838 → {heuristic_result['optimal_base_stock_level']:,.0f} units")
print()
print("SIMULATION VERIFICATION RESULTS:")
print(f"• Service Level Achieved: {simulation_result['service_level']*100:.2f}% (target: 95%)")
print(f"• Cost Verification: {cost_diff_pct:.1f}% difference from analytical")
print(f"• Average Inventory: {simulation_result['avg_inventory']:,.0f} units")
print(f"• Stockout Periods: {simulation_result['stockout_periods']}/52 weeks")
print()
print("KEY HEURISTIC INSIGHTS:")
print("• Optimal review period differs from standard weekly assumption")
print("• Cost savings achieved through reduced ordering frequency")
print("• Trade-off between ordering costs and inventory holding")
print("• Simulation confirms analytical calculations")
print()
print("PRACTICAL ADVANTAGES:")
print("• Flexible review period optimization")
print("• Robust through simulation verification")
print("• Suitable for real-world implementation")
print("• Handles discrete decision variables effectively")
print()
print("WHEN TO USE HEURISTIC APPROACH:")
print("• When review periods are constrained or flexible")
print("• When simulation verification is required")
print("• For practical implementation in operational systems")
print("• When robustness is preferred over theoretical optimality")